In [ ]:
import os
import asyncio
import sys
from pathlib import Path

# Auto-load .env file
from dotenv import load_dotenv

# Load .env and add project root to path
load_dotenv(Path('..') / '.env')
sys.path.insert(0, str(Path('..').resolve()))

print('Environment loaded.')

from src.bot import TradingBot
from src.config import Config

In [ ]:
private_key = os.environ.get("POLY_PRIVATE_KEY")

USE BOT

In [ ]:
# Initialize the bot from environment
# config = Config.from_env()
config = Config.load_with_env("config.yaml")
bot = TradingBot(config=config, private_key=private_key)

print(f"\nBot initialized with Safe: {bot.config.safe_address}")
print(f"Gasless mode: {bot.config.use_gasless}")

In [ ]:
# get open orders
orders = await bot.get_open_orders()
orders

In [ ]:
# get recent trades
trades = await bot.get_trades(limit=5)
for trade in trades[:3]:  # Show first 3
    # print time and then the trade
    print()
    print(trade)
    print("---")

In [ ]:
import requests
from datetime import datetime, timezone
import json

now = datetime.now(timezone.utc)
ts = int(now.timestamp())
ts = (ts // 300) * 300  # round to current 5-min window

for offset in [0, 300, -300]:
    slug = f'btc-updown-5m-{ts + offset}'
    r = requests.get(f'https://gamma-api.polymarket.com/markets/slug/{slug}')
    ids = json.loads(r.json()['clobTokenIds'])
    id_up = ids[0] # current id for up bid

id_up

In [ ]:
price = 0.30
result = await bot.place_order(
    token_id=id_up,
    price=price,   # well below market (0.485) — won't fill, safe to test
    size=1/price,     # minimum order size is 5 shares
    side="BUY"
)
print(f"Order result: {result.success}, {result.order_id}")
